# Day 09 - GroupBy Advanced, Pivot Tables and Crosstabs  (SOLUTIONS)

**Goal:** Multi-column summaries and business-style tables.

**Quick recap (read for 2 minutes):**

- `.agg(['mean','median','std'])` several stats at once
- Named aggregation: `.agg(users=('user_id','count'), avg_age=('age','mean'))`
- `pd.pivot_table(df, index=, columns=, values=, aggfunc=)`
- `pd.crosstab(a, b)`; `normalize='index'` gives row percentages
- `.transform('mean')` returns a value for every original row

*There are many valid ways to solve each question. If your answer gives the same result, it is correct.*

### Warm-up (5 min)
Without looking, redo Questions 1 to 3 from **Day 08** in a scratch cell below.

### Setup (run this first)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("spotify_user_behv.csv")

### Q1. For each subscription type, show the mean, median, std and count of listening hours.

In [ ]:
df.groupby("subscription_type")["avg_listening_hours_per_week"].agg(["mean", "median", "std", "count"]).round(2)

### Q2. Per country, build a table with `users`, `avg_age` and `avg_hours` using named aggregation.

In [ ]:
df.groupby("country").agg(users=("user_id", "count"), avg_age=("age", "mean"), avg_hours=("avg_listening_hours_per_week", "mean")).round(2)

### Q3. Average hours by country AND subscription type (group by two columns).

In [ ]:
df.groupby(["country", "subscription_type"])["avg_listening_hours_per_week"].mean().round(2)

### Q4. Make a pivot table: rows = country, columns = subscription type, values = average hours.

In [ ]:
pd.pivot_table(df, index="country", columns="subscription_type", values="avg_listening_hours_per_week", aggfunc="mean").round(2)

### Q5. Crosstab of subscription type vs ad interaction (counts).

In [ ]:
pd.crosstab(df["subscription_type"], df["ad_interaction"])

### Q6. Crosstab of subscription type vs ad conversion as row percentages.

In [ ]:
(pd.crosstab(df["subscription_type"], df["ad_conversion_to_subscription"], normalize="index") * 100).round(1)

### Q7. Churn rate (% of `inactive_3_months_flag`) per subscription type, highest first.

In [ ]:
(df.groupby("subscription_type")["inactive_3_months_flag"].mean() * 100).round(1).sort_values(ascending=False)

### Q8. Add `country_avg_hours`, the average hours of each user's country, using `transform`.

In [ ]:
df["country_avg_hours"] = df.groupby("country")["avg_listening_hours_per_week"].transform("mean")
df[["country", "avg_listening_hours_per_week", "country_avg_hours"]].head()

### Q9. Add `diff_from_country` = user hours - country average. How many users are above their country average?

In [ ]:
df["country_avg_hours"] = df.groupby("country")["avg_listening_hours_per_week"].transform("mean")
df["diff_from_country"] = df["avg_listening_hours_per_week"] - df["country_avg_hours"]
(df["diff_from_country"] > 0).sum()

### Q10. Find the top 3 favorite genres in each country by number of users.

In [ ]:
g = df.groupby(["country", "favorite_genre"]).size().reset_index(name="n")
g.sort_values(["country", "n"], ascending=[True, False]).groupby("country").head(3)